In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Transpile OOF Multinomial Logistic Meta-Learner to Pure C with `m2cgen` (`models/transpile_soft_pipeline_to_c.ipynb`)

This notebook imports the optimal **OOF Multinomial Logistic Regression Meta-Learner Bundle** (`deploy/oof_multinomial_logistic_meta_learner.rds`) and uses **`m2cgen`** to transpile the LightGBM sub-models and Meta-Learner directly into zero-dependency **pure C code** (`deploy/triage_pipeline.c` & `deploy/triage_pipeline.h`).

### Transpiled Components (`m2cgen`)
1. **Layer 1 Booster**: `predict_layer1(x)`
2. **Layer 2 Booster**: `predict_layer2(x)`
3. **Layer 3A Booster**: `predict_layer3a(x)`
4. **Layer 3B Booster**: `predict_layer3b(x)`
5. **Multinomial Logistic Meta-Learner**: `predict_meta_logistic(probs)`
6. **Full Embedded C API**: `predict_triage(const TriageInput* input)`

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load deploy/oof_multinomial_logistic_meta_learner.rds & Export Model Artifacts
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(lightgbm)
})
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
rds_path <- file.path(deploy_dir, "oof_multinomial_logistic_meta_learner.rds")
if (!file.exists(rds_path)) {
  stop(paste("Error:", rds_path, "does not exist! Please run models/train_oof_logistic_regression_stacking.ipynb first."))
}
bundle <- readRDS(rds_path)
# Save LightGBM Boosters as Model Text Files for Python / m2cgen
lgb.save(bundle$l1_model,  file.path(deploy_dir, "l1_booster.txt"))
saveRDS(bundle$l1_model,   file.path(deploy_dir, "lightgbm_layer1_esi1_model.rds"))
lgb.save(bundle$l2_model,  file.path(deploy_dir, "l2_booster.txt"))
saveRDS(bundle$l2_model,   file.path(deploy_dir, "rf_esi23_esi45_extreme_model.rds"))
lgb.save(bundle$l3a_model, file.path(deploy_dir, "l3a_booster.txt"))
saveRDS(bundle$l3a_model,  file.path(deploy_dir, "lightgbm_esi23_model.rds"))
lgb.save(bundle$l3b_model, file.path(deploy_dir, "l3b_booster.txt"))
saveRDS(bundle$l3b_model,  file.path(deploy_dir, "lightgbm_esi45_model.rds"))
# Export Meta-Learner Parameters and Scaler as JSON
meta_info <- list(
  intercepts  = as.numeric(bundle$intercepts),
  coef_matrix = lapply(1:nrow(bundle$coef_matrix), function(i) as.numeric(bundle$coef_matrix[i, ])),
  scaler      = list(means = as.list(bundle$scaler$means), sds = as.list(bundle$scaler$sds), cols = bundle$scaler$cols),
  feature_cols= bundle$feature_cols
)
write(jsonlite::toJSON(meta_info, auto_unbox = TRUE), file.path(deploy_dir, "meta_learner_info.json"))
cat("Successfully loaded deploy/oof_multinomial_logistic_meta_learner.rds and exported booster & JSON artifacts!\n")

In [ ]:
# ---------------------------------------------------------
# Step 2: Transpile Models to C using m2cgen (LGBMClassifier Wrappers)
# ---------------------------------------------------------
import os
import sys
import json
import numpy as np
try:
    import m2cgen as m2c
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "m2cgen"])
    import m2cgen as m2c
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
deploy_dir = '../deploy' if os.path.exists('../deploy') else 'deploy'
with open(os.path.join(deploy_dir, 'meta_learner_info.json')) as f:
    meta_info = json.load(f)
intercepts  = np.array(meta_info['intercepts'], dtype=np.float64)
coef_matrix = np.array(meta_info['coef_matrix'], dtype=np.float64)
scaler_means = meta_info['scaler']['means']
scaler_sds   = meta_info['scaler']['sds']
scaler_cols  = meta_info['scaler']['cols']
# Helper to wrap raw lightgbm.Booster into lightgbm.LGBMClassifier for m2cgen
def wrap_booster_for_m2cgen(booster_file, n_feats=38):
    booster = lgb.Booster(model_file=booster_file)
    clf = lgb.LGBMClassifier()
    clf._booster = booster
    clf.booster_ = booster
    clf._n_classes = 2
    clf.classes_ = np.array([0, 1])
    clf._n_features_in = n_feats
    clf.n_features_in_ = n_feats
    clf._objective = 'binary'
    return clf
clf_l1  = wrap_booster_for_m2cgen(os.path.join(deploy_dir, 'l1_booster.txt'))
clf_l2  = wrap_booster_for_m2cgen(os.path.join(deploy_dir, 'l2_booster.txt'))
clf_l3a = wrap_booster_for_m2cgen(os.path.join(deploy_dir, 'l3a_booster.txt'))
clf_l3b = wrap_booster_for_m2cgen(os.path.join(deploy_dir, 'l3b_booster.txt'))
# Reconstruct Fitted Scikit-Learn Multinomial Logistic Regression Meta-Learner
meta_logreg = LogisticRegression(multi_class='multinomial', class_weight='balanced')
meta_logreg.classes_ = np.array([1, 2, 3, 4, 5])
meta_logreg.intercept_ = intercepts
meta_logreg.coef_ = coef_matrix
meta_logreg.n_features_in_ = 5
# Transpile Models to Pure C using m2cgen
c_code_l1  = m2c.export_to_c(clf_l1,  function_name='predict_layer1')
c_code_l2  = m2c.export_to_c(clf_l2,  function_name='predict_layer2')
c_code_l3a = m2c.export_to_c(clf_l3a, function_name='predict_layer3a')
c_code_l3b = m2c.export_to_c(clf_l3b, function_name='predict_layer3b')
# Transpile Meta-Learner to C via m2cgen
c_code_meta = m2c.export_to_c(meta_logreg, function_name='predict_meta_logistic')
print("m2cgen Transpilation Completed Successfully for Sub-Models and Meta-Learner!")

In [ ]:
# ---------------------------------------------------------
# Step 3: Assemble Pure C Header & Source Code (deploy/triage_pipeline.h & triage_pipeline.c)
# ---------------------------------------------------------
header_content = """#ifndef TRIAGE_PIPELINE_H
#define TRIAGE_PIPELINE_H

#ifdef __cplusplus
extern "C" {
#endif

// Raw Input Features (15 Values)
typedef struct {
    float age;
    float cc_breathingdifficulty;
    float gender;
    float triage_vital_hr;
    float triage_vital_sbp;
    float triage_vital_rr;
    float triage_vital_o2;
    float pulse_min;
    float resp_min;
    float spo2_min;
    float sbp_min;
    float pulse_max;
    float resp_max;
    float spo2_max;
    float sbp_max;
} TriageInput;
// 5-Class Output Probabilities and Predicted ESI Level
typedef struct {
    float probs[5]; // Index 0..4 maps to ESI 1..5
    int predicted_esi; // 1..5
} TriageOutput;
// Primary C Pipeline Entry Point
TriageOutput predict_triage(const TriageInput* input);
#ifdef __cplusplus
}
#endif
#endif // TRIAGE_PIPELINE_H
"""
# Build Scaler C Code
feature_names = [
    'age', 'cc_breathingdifficulty', 'gender', 'triage_vital_hr', 'triage_vital_sbp', 'triage_vital_rr', 'triage_vital_o2',
    'pulse_min', 'resp_min', 'spo2_min', 'sbp_min', 'pulse_max', 'resp_max', 'spo2_max', 'sbp_max',
    'is_dyspnea_total', 'is_dyspnea_moderate', 'is_bradypnea', 'is_tachypnea', 'is_hypotension', 'is_hypertension',
    'is_bradycardia_total', 'is_bradycardia_moderate', 'is_tachycardia_total', 'is_tachycardia_moderate',
    'hr_range', 'rr_range', 'spo2_range', 'sbp_range',
    'shock_index', 'hr_mid_to_triage', 'sbp_mid_to_triage', 'rr_mid_to_triage', 'spo2_mid_to_triage',
    'rox_index', 'spo2_drop_ratio', 'hr_instability_ratio', 'bif'
]
scaling_lines = []
for idx, f_name in enumerate(feature_names):
    if f_name in scaler_cols:
        m = scaler_means[f_name]
        s = scaler_sds[f_name]
        scaling_lines.append(f"    x[{idx}] = (x[{idx}] - {m:.8f}f) / {s:.8f}f;")
scaling_c_str = "\n".join(scaling_lines)
c_source_content = f"""#include <math.h>
#include "triage_pipeline.h"
// --- Transpiled Sub-Models (m2cgen) ---
{c_code_l1}
{c_code_l2}
{c_code_l3a}
{c_code_l3b}
// --- Transpiled Multinomial Logistic Meta-Learner (m2cgen) ---
{c_code_meta}
TriageOutput predict_triage(const TriageInput* in) {{
    TriageOutput out;
    double x[38];
    
    // 1. Raw Feature Extraction
    x[0]  = (double)in->age;
    x[1]  = (double)in->cc_breathingdifficulty;
    x[2]  = (double)in->gender;
    x[3]  = (double)in->triage_vital_hr;
    x[4]  = (double)in->triage_vital_sbp;
    x[5]  = (double)in->triage_vital_rr;
    x[6]  = (double)in->triage_vital_o2;
    x[7]  = (double)in->pulse_min;
    x[8]  = (double)in->resp_min;
    x[9]  = (double)in->spo2_min;
    x[10] = (double)in->sbp_min;
    x[11] = (double)in->pulse_max;
    x[12] = (double)in->resp_max;
    x[13] = (double)in->spo2_max;
    x[14] = (double)in->sbp_max;
    
    // 2. Clinical Flags & Vital Ranges
    x[15] = (in->triage_vital_o2 < 90.0f) ? 1.0 : 0.0; // is_dyspnea_total
    x[16] = (in->triage_vital_o2 > 90.0f && in->triage_vital_o2 < 94.0f) ? 1.0 : 0.0; // is_dyspnea_moderate
    x[17] = (in->triage_vital_rr < 10.0f) ? 1.0 : 0.0; // is_bradypnea
    x[18] = (in->triage_vital_rr > 30.0f) ? 1.0 : 0.0; // is_tachypnea
    x[19] = (in->triage_vital_sbp <= 90.0f) ? 1.0 : 0.0; // is_hypotension
    x[20] = (in->triage_vital_sbp > 220.0f) ? 1.0 : 0.0; // is_hypertension
    x[21] = (in->triage_vital_hr < 40.0f) ? 1.0 : 0.0; // is_bradycardia_total
    x[22] = (in->triage_vital_hr > 40.0f && in->triage_vital_hr < 60.0f) ? 1.0 : 0.0; // is_bradycardia_moderate
    x[23] = (in->triage_vital_hr > 150.0f) ? 1.0 : 0.0; // is_tachycardia_total
    x[24] = (in->triage_vital_hr > 100.0f && in->triage_vital_hr < 150.0f) ? 1.0 : 0.0; // is_tachycardia_moderate
    x[25] = (double)(in->pulse_max - in->pulse_min); // hr_range
    x[26] = (double)(in->resp_max - in->resp_min);   // rr_range
    x[27] = (double)(in->spo2_max - in->spo2_min);   // spo2_range
    x[28] = (double)(in->sbp_max - in->sbp_min);     // sbp_range
    
    // 3. Advanced Clinical Indicators
    x[29] = (double)(in->triage_vital_hr / ((in->triage_vital_sbp == 0.0f) ? 1.0f : in->triage_vital_sbp)); // shock_index
    x[30] = (double)(in->triage_vital_hr - x[25]); // hr_mid_to_triage
    x[31] = (double)(in->triage_vital_sbp - x[28]); // sbp_mid_to_triage
    x[32] = (double)(in->triage_vital_rr - x[26]); // rr_mid_to_triage
    x[33] = (double)(in->triage_vital_o2 - x[27]); // spo2_mid_to_triage
    x[34] = (double)(in->triage_vital_o2 / ((in->triage_vital_rr == 0.0f) ? 1.0f : in->triage_vital_rr)); // rox_index
    x[35] = (double)(x[27] / ((in->spo2_max == 0.0f) ? 1.0f : in->spo2_max)); // spo2_drop_ratio
    x[36] = (double)(x[25] / (in->triage_vital_hr + 1.0f)); // hr_instability_ratio
    x[37] = (double)((in->triage_vital_rr / ((in->triage_vital_o2 == 0.0f) ? 1.0f : in->triage_vital_o2)) * 100.0f); // bif
    
    // 4. Feature Standardization Scaling
{scaling_c_str}
    
    // 5. Evaluate Sub-Models (m2cgen)
    double p1  = predict_layer1(x);
    double p2  = predict_layer2(x);
    double p3a = predict_layer3a(x);
    double p3b = predict_layer3b(x);
    
    // 6. Base 5-Class Soft Joint Probabilities
    double base_probs[5];
    base_probs[0] = p1;
    base_probs[1] = (1.0 - p1) * p2 * p3a;
    base_probs[2] = (1.0 - p1) * p2 * (1.0 - p3a);
    base_probs[3] = (1.0 - p1) * (1.0 - p2) * p3b;
    base_probs[4] = (1.0 - p1) * (1.0 - p2) * (1.0 - p3b);
    
    // 7. Evaluate Multinomial Logistic Meta-Learner (m2cgen)
    double meta_probs[5];
    predict_meta_logistic(base_probs, meta_probs);
    
    int best_esi = 1;
    double max_p = meta_probs[0];
    out.probs[0] = (float)meta_probs[0];
    
    for (int k = 1; k < 5; k++) {{
        out.probs[k] = (float)meta_probs[k];
        if (meta_probs[k] > max_p) {{
            max_p = meta_probs[k];
            best_esi = k + 1;
        }}
    }}
    out.predicted_esi = best_esi;
    return out;
}}
"""
with open(os.path.join(deploy_dir, 'triage_pipeline.h'), 'w') as f:
    f.write(header_content)
with open(os.path.join(deploy_dir, 'triage_pipeline.c'), 'w') as f:
    f.write(c_source_content)
print(f"Pure C Pipeline Source Transpiled via m2cgen successfully! Deliverables: {os.path.join(deploy_dir, 'triage_pipeline.c')} & {os.path.join(deploy_dir, 'triage_pipeline.h')}")